In [1]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import ssl
import os
import urllib.request

# 1. แก้ไขปัญหา SSL (ต้องอยู่ก่อน import easyocr)
ssl._create_default_https_context = ssl._create_unverified_context

# สร้าง custom opener เพื่อ bypass SSL verification สำหรับ urllib
opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl._create_unverified_context()))
urllib.request.install_opener(opener)

import cv2
import numpy as np
import easyocr
import numpy as np
import imutils
from ultralytics import YOLO

# 2. Setup EasyOCR
print("กำลังโหลด Model... (อาจใช้เวลาสักครู่ในการดาวน์โหลดครั้งแรก)")
# เปลี่ยน gpu=True หากเครื่องคุณมี NVIDIA GPU และลง CUDA ไว้
reader = easyocr.Reader(['th', 'en'], gpu=False)

# โหลดโมเดล YOLO ที่คุณเทรนเองสำหรับตรวจจับป้ายทะเบียน
print("กำลังโหลดโมเดล vehicle_detector.pt...")
model = YOLO('vehicle_detector.pt')

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def perspective_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect
    width = max(int(np.linalg.norm(br - bl)), int(np.linalg.norm(tr - tl)))
    height = max(int(np.linalg.norm(tr - br)), int(np.linalg.norm(tl - bl)))
    dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(image, M, (width, height))

# 3. เริ่มต้นเปิดกล้อง
cap = cv2.VideoCapture(1)

# ปรับความละเอียดกล้องให้เหมาะสม
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_H88888EIGHT, 480)

print("ระบบพร้อมทำงาน! กด 'q' เพื่อปิดโปรแกรม")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    display_frame = frame.copy()
    
    # ใช้ YOLO ตรวจจับป้ายทะเบียน
    results = model(frame, conf=0.5)
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            
            # วาดกรอบรอบป้าย
            cv2.rectangle(display_frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            
            try:
                # ตัดรูปป้ายทะเบียน
                plate_img = frame[int(y1):int(y2), int(x1):int(x2)]
                plate_gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
                
                # อ่าน OCR
                ocr_results = reader.readtext(plate_gray)
                
                if len(ocr_results) > 0:
                    # เรียงบรรทัดจากบนลงล่าง
                    ocr_results.sort(key=lambda x: x[0][0][1])
                    all_text = [res[1] for res in ocr_results if res[2] > 0.15]
                    
                    # Logic แยกประเภทรถยนต์/มอเตอร์ไซค์ ตามจำนวนข้อความ
                    if len(all_text) >= 3:
                        v_type = "Motorcycle"
                        plate = f"{all_text[0]} {all_text[-1]}" if len(all_text) >= 2 else "".join(all_text)
                        province = all_text[1] if len(all_text) >= 3 else "Unknown"
                    else:
                        v_type = "Car"
                        plate = all_text[0] if len(all_text) >= 1 else ""
                        province = all_text[1] if len(all_text) >= 2 else "Unknown"

                    # แสดงผลที่ Terminal
                    print(f"[{v_type}] ทะเบียน: {plate} | จังหวัด: {province}")
                    
                    # เขียนข้อความลงบนจอหลัก
                    cv2.putText(display_frame, f"{v_type}: {plate}", (int(x1), int(y1) - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            except Exception as e:
                pass

    # แสดงภาพจากกล้อง
    cv2.imshow("AI License Plate Detection", display_frame)

    # กด q เพื่อออก
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Using CPU. Note: This module is much faster with a GPU.


กำลังโหลด Model... (อาจใช้เวลาสักครู่ในการดาวน์โหลดครั้งแรก)
กำลังโหลดโมเดล vehicle_detector.pt...
ระบบพร้อมทำงาน! กด 'q' เพื่อปิดโปรแกรม

0: 480x640 (no detections), 129.6ms
Speed: 10.2ms preprocess, 129.6ms inference, 5.5ms postprocess per image at shape (1, 3, 480, 640)


error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'
